# 20260923 Visualize Trials

Use this notebook for trial-level exploration: individual trial trajectories, Bpod/video-port event timing, success/unsuccess comparisons, trajectory labels, and optional neural activity aligned to trial phase events.

## Imports

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.pipeline import default_aligned_session_path, default_cue_events_path, default_trials_path
from preprocess_functions import plot

## Config

In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_cells.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"
if not MANIFEST.exists():
    MANIFEST = Path("data_paths/RSC_PPC_Cohort1_paths.xlsx")

SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
RECORDING_ID = None
FPS = 30.0

LABELS_CSV = OUTPUT_ROOT / "trial_classification" / "trial_labels.csv"

# Choose how success/correctness should be defined.
# "cue" uses actual arrow-key cue labels -> cue directions -> expected port.
# "trajectory" uses trajectory label side -> expected port.
EXPECTED_PORT_MODE = "cue"
CUE_ARROW_KEY_TO_DIRECTION = {
    "LEFT": "left", "ARROWLEFT": "left", "LEFTARROW": "left", "←": "left",
    "RIGHT": "right", "ARROWRIGHT": "right", "RIGHTARROW": "right", "→": "right",
    "DOWN": "down", "ARROWDOWN": "down", "DOWNARROW": "down", "↓": "down",
    "UP": "up", "ARROWUP": "up", "UPARROW": "up", "↑": "up",
}
CUE_DIRECTION_TO_EXPECTED_PORT = {"left": 1, "right": 2, "up": 3, "down": 4}
SIDE_TO_EXPECTED_PORT = {"L": 1, "R": 2}
LABEL_TO_TRAJECTORY_SIDE = {
    "left_small_loop": "L",
    "left_inverse_small_loop": "L",
    "left_big_loop": "L",
    "right_small_loop": "R",
    "right_inverse_small_loop": "R",
    "right_big_loop": "R",
    "non_characteristic": None,
}

TRIAL_IDX = None
NEURAL_EVENT_SOURCE = "cue_start"  # cue_start, trial_start, trial_end, or port_onset
NEURAL_EVENT_WINDOW_S = 2.0
NEURAL_EVENT_CELLS = None  # None uses the first 3 cell_* columns, if present

## Load Recording And Trial Tables

In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "mouse_id": record.mouse_id,
        "trial_type": record.trial_type,
        "aligned_csv": str(default_aligned_session_path(OUTPUT_ROOT, record)),
        "trials_csv": str(default_trials_path(OUTPUT_ROOT, record)),
    }
    for record in records
])
display(records_df)

In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        return records[0]
    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")


record = choose_record(records, RECORDING_ID)
aligned_csv = default_aligned_session_path(OUTPUT_ROOT, record)
trials_csv = default_trials_path(OUTPUT_ROOT, record)
cue_events_csv = default_cue_events_path(OUTPUT_ROOT, record)

if not aligned_csv.exists():
    raise FileNotFoundError(f"Missing aligned CSV: {aligned_csv}")
if not trials_csv.exists():
    raise FileNotFoundError(f"Missing trials CSV. Run trial segmentation first: {trials_csv}")

df = pd.read_csv(aligned_csv)
trials_df = pd.read_csv(trials_csv)
cue_events_df = pd.read_csv(cue_events_csv) if cue_events_csv.exists() else pd.DataFrame()

print("recording_id:", record.recording_id)
print("aligned CSV:", aligned_csv)
print("trials CSV:", trials_csv)
print("aligned shape:", df.shape)
display(trials_df.head())

## Merge Trajectory Labels

In [ ]:
if LABELS_CSV.exists():
    labels_df = pd.read_csv(LABELS_CSV)
    labels_df["trial_idx"] = pd.to_numeric(labels_df["trial_idx"], errors="coerce").astype("Int64")
    label_cols = ["recording_id", "trial_idx", "label"]
    optional_label_cols = [col for col in ["notes", "label_source", "pred_label", "pred_conf"] if col in labels_df.columns]
    trials_labeled = trials_df.merge(
        labels_df[label_cols + optional_label_cols].rename(columns={"label": "trajectory_label"}),
        on=["recording_id", "trial_idx"],
        how="left",
    )
else:
    labels_df = pd.DataFrame()
    trials_labeled = trials_df.copy()
    trials_labeled["trajectory_label"] = pd.NA
    print("No trajectory label file yet:", LABELS_CSV)

display(trials_labeled.head())

## Port Onsets And Success/Correctness

In [ ]:
event_cols = plot.event_active_columns(df)
print("Bpod/video port active columns:", event_cols)


def active_onset_events(aligned_df, active_cols):
    rows = []
    for col in active_cols:
        active = pd.to_numeric(aligned_df[col], errors="coerce").fillna(0).astype(bool).to_numpy()
        onsets = np.flatnonzero(active & ~np.r_[False, active[:-1]])
        if "_port_" in col:
            port = int(col.split("_port_")[1].split("_")[0])
        else:
            port = np.nan
        for frame in onsets:
            rows.append({
                "port": port,
                "event_frame": int(frame),
                "event_ts": aligned_df["global_ts"].iloc[frame] if "global_ts" in aligned_df.columns else pd.NaT,
                "event_col": col,
            })
    return pd.DataFrame(rows)


def first_port_for_trial(trial_row, port_events):
    if port_events.empty:
        return pd.Series({"active_port": np.nan, "active_port_frame": np.nan, "active_port_ts": pd.NaT})
    hits = port_events[
        port_events["event_frame"].between(
            int(trial_row["start_frame"]),
            int(trial_row["end_frame"]),
            inclusive="both",
        )
    ]
    if hits.empty:
        return pd.Series({"active_port": np.nan, "active_port_frame": np.nan, "active_port_ts": pd.NaT})
    first = hits.sort_values("event_frame").iloc[0]
    return pd.Series({
        "active_port": first["port"],
        "active_port_frame": first["event_frame"],
        "active_port_ts": first["event_ts"],
    })


port_events = active_onset_events(df, event_cols)
port_by_trial = trials_labeled.apply(first_port_for_trial, axis=1, port_events=port_events)
trial_metrics = plot.compute_trial_behavior_metrics(df, trials_labeled, fps=FPS, event_cols=event_cols)
trial_metrics = pd.concat([trial_metrics, port_by_trial], axis=1)

def cue_arrow_to_direction(value):
    if pd.isna(value):
        return np.nan
    cue = str(value).strip()
    compact = cue.replace(" ", "").replace("_", "").replace("-", "").upper()
    direction = CUE_ARROW_KEY_TO_DIRECTION.get(compact) or CUE_ARROW_KEY_TO_DIRECTION.get(cue)
    if direction is not None:
        return direction
    cue_lower = cue.lower()
    if cue_lower in CUE_DIRECTION_TO_EXPECTED_PORT:
        return cue_lower
    return np.nan

trial_metrics["trajectory_side"] = trial_metrics["trajectory_label"].map(LABEL_TO_TRAJECTORY_SIDE)
if EXPECTED_PORT_MODE == "cue":
    cue_values = trial_metrics.get("cue_key", pd.Series(index=trial_metrics.index, dtype=object))
    trial_metrics["cue_direction"] = cue_values.map(cue_arrow_to_direction)
    trial_metrics["expected_port"] = trial_metrics["cue_direction"].map(CUE_DIRECTION_TO_EXPECTED_PORT)
elif EXPECTED_PORT_MODE == "trajectory":
    trial_metrics["expected_port"] = trial_metrics["trajectory_side"].map(SIDE_TO_EXPECTED_PORT)
else:
    raise ValueError("EXPECTED_PORT_MODE must be 'cue' or 'trajectory'")

trial_metrics["correct_trajectory"] = trial_metrics["trajectory_side"].notna()
trial_metrics["correct_port"] = trial_metrics["active_port"] == trial_metrics["expected_port"]
trial_metrics["success"] = trial_metrics["correct_trajectory"] & trial_metrics["correct_port"]

display(trial_metrics.head())
display(trial_metrics["success"].value_counts(dropna=False).rename("n").to_frame())

## One Trial: Trajectory, Events, Optional Neural

In [ ]:
if TRIAL_IDX is None:
    TRIAL_IDX = int(trial_metrics["trial_idx"].iloc[0])

cell_cols = [col for col in df.columns if col.startswith("registered_cell_")]
if not cell_cols:
    cell_cols = [col for col in df.columns if col.startswith("cell_")]
if NEURAL_EVENT_CELLS is None:
    neural_trial_cells = cell_cols[:1]
else:
    neural_trial_cells = [col for col in NEURAL_EVENT_CELLS if col in df.columns]

print("selected trial:", TRIAL_IDX)
print("optional neural cells for trial plot:", neural_trial_cells)

trial_row = trial_metrics.loc[trial_metrics["trial_idx"] == TRIAL_IDX].iloc[0]
plot.plot_trial_behavior_events(
    df,
    trial_row,
    event_cols=event_cols,
    neural_cols=neural_trial_cells,
)

## Compare Correct/Incorrect Or Label Groups

In [ ]:
GROUP_COL = "success"  # Try "success", "trajectory_label", "active_port", "expected_port", or "cue_key".

plot.plot_trial_metric_summary(trial_metrics, group_col=GROUP_COL)
plot.plot_trial_trajectories_by_group(
    df,
    trial_metrics,
    group_col=GROUP_COL,
    max_trials_per_group=25,
)

## Optional Neural Event/Phase Alignment

In [ ]:
cell_cols = [col for col in df.columns if col.startswith("registered_cell_")]
if not cell_cols:
    cell_cols = [col for col in df.columns if col.startswith("cell_")]
if NEURAL_EVENT_CELLS is None:
    neural_event_cells = cell_cols[:3]
else:
    neural_event_cells = [col for col in NEURAL_EVENT_CELLS if col in df.columns]

event_tables = {}
if "cue_start_frame" in trial_metrics.columns:
    event_tables["cue_start"] = (trial_metrics.dropna(subset=["cue_start_frame"]), "cue_start_frame", None)
event_tables["trial_start"] = (trial_metrics, "start_frame", None)
event_tables["trial_end"] = (trial_metrics, "end_frame", None)
if not port_events.empty:
    event_tables["port_onset"] = (port_events, "event_frame", "event_ts")

print("available event sources:", list(event_tables))
print("optional neural cells:", neural_event_cells)

if neural_event_cells and NEURAL_EVENT_SOURCE in event_tables:
    events_for_neural, frame_col, ts_col = event_tables[NEURAL_EVENT_SOURCE]
    if not events_for_neural.empty:
        plot.plot_event_aligned_neural(
            df,
            events_for_neural,
            cell_cols=neural_event_cells,
            event_frame_col=frame_col,
            event_ts_col=None,
            window_s=NEURAL_EVENT_WINDOW_S,
            fps=FPS,
        )
else:
    print("No neural event-aligned plot: either no cell_* columns, or the requested event source is missing.")

## Count Correct Trials

In [ ]:
summary = (
    trial_metrics
    .groupby(["recording_id", "success"], dropna=False)
    .size()
    .rename("n_trials")
    .reset_index()
)
display(summary)

if {"correct_trajectory", "correct_port"}.issubset(trial_metrics.columns):
    display(pd.crosstab(trial_metrics["correct_trajectory"], trial_metrics["correct_port"], margins=True))